# LAB 33 — Flood Response Pipeline: Від Sequential Collapse до Hybrid Architecture

## Disaster-Driven Engineering Simulation

**Модуль 4 · Network & Concurrent Systems · Автор: Viktor Nikoriak**

---

```
╔══════════════════════════════════════════════════════════════╗
║  🚨  НАДЗВИЧАЙНА СИТУАЦІЯ  🚨                               ║
║                                                              ║
║  КАРПАТИ. РАПТОВА ПОВІНЬ.                                    ║
║  500 SENTINEL-1 TILES завантажуються кожні 6 хвилин.         ║
║  ДСНС очікує flood map.                                      ║
║  Люди в небезпеці.                                           ║
║                                                              ║
║  Твоя система має обробити всі tiles ДО приходу повені.      ║
║  У тебе є 30 хвилин.                                         ║
╚══════════════════════════════════════════════════════════════╝
```

---

## 5 архітектурних рішень, які ти пройдеш:

| Етап | Підхід | Результат |
|------|--------|----------|
| 1 | Sequential processing | Система не встигає - повінь приходить раніше |
| 2 | Threading (CPU-bound) | GIL блокує - все одно один core |
| 3 | Multiprocessing | Справжній паралелізм - але нова проблема |
| 4 | Наївний ProcessPool | Pickle overhead вбиває pipeline |
| 5 | Hybrid Architecture | Правильне рішення - система рятує людей |

> **Симуляція:** затримки та overhead - реалістичні моделі продакшн поведінки.

## Ініціалізація Flood Response Infrastructure

In [ ]:
import time
import random
import os
import numpy as np
import threading
import multiprocessing
import concurrent.futures
from dataclasses import dataclass


# Конфігурація flood event
TOTAL_TILES = 40
FLOOD_DEADLINE_SEC = 15
TILE_PROCESS_TIME = 0.25   # CPU-bound SAR analysis (секунди)
TILE_DOWNLOAD_TIME = 0.12  # I/O-bound download (секунди)
CPU_CORES = os.cpu_count() or 4

@dataclass
class SentinelTile:
    tile_id: str
    region: str
    data_size_mb: float
    raster_data: list

def create_tile(tile_id: int) -> SentinelTile:
    regions = ['Закарпаття', 'Прикарпаття', 'Чернівці', 'Івано-Франківськ']
    return SentinelTile(
        tile_id=f'S1A_IW_{tile_id:04d}',
        region=random.choice(regions),
        data_size_mb=round(random.uniform(50, 120), 1),
        raster_data=list(range(500))
    )

ALL_TILES = [create_tile(i) for i in range(TOTAL_TILES)]

print('Flood Response System ініціалізовано')
print(f'CPU cores: {CPU_CORES}')
print(f'Tiles: {TOTAL_TILES}')
print(f'Deadline: {FLOOD_DEADLINE_SEC}s (симуляція)')
print()
print(f'Sequential estimate: {TOTAL_TILES * TILE_PROCESS_TIME:.1f}s')
print(f'Parallel estimate ({CPU_CORES} cores): {TOTAL_TILES * TILE_PROCESS_TIME / CPU_CORES:.1f}s')

---

# ЕТАП 1 - Sequential Collapse

```
┌─────────────────────────────────────────────────────────┐
│  ДИСПЕТЧЕР: 'Система, скільки tiles оброблено?'         │
│  СИСТЕМА:   'Обробляю tile 3 з 500...'                  │
│  ДИСПЕТЧЕР: 'Повінь прийде через 10 хвилин!'            │
│  СИСТЕМА:   '...tile 4 з 500...'                        │
└─────────────────────────────────────────────────────────┘
```

## Наївна реалізація: `for tile in tiles: process(tile)`

**Передбач:** скільки часу займе обробка 12 tiles?  
- Tile processing: 0.25s кожен  
- Download: 0.12s кожен  
- Save: 0.05s кожен  
- TOTAL per tile: 0.42s  
- **Sequential: 12 x 0.42 = 5.04s**

In [ ]:
# Stage functions
def download_tile(tile):
    # I/O-bound: HTTP download from Copernicus Hub
    time.sleep(TILE_DOWNLOAD_TIME)
    return tile

def generate_flood_mask(tile):
    # CPU-bound: SAR backscatter threshold analysis
    time.sleep(TILE_PROCESS_TIME)
    flood_pixels = sum(1 for v in tile.raster_data if v % 7 < 2)
    return {
        'tile_id': tile.tile_id,
        'region': tile.region,
        'flood_pixels': flood_pixels,
        'flood_pct': round(flood_pixels / len(tile.raster_data) * 100, 1)
    }

def save_to_postgis(result):
    # I/O-bound: INSERT into PostGIS
    time.sleep(0.05)

# SEQUENTIAL PIPELINE
demo_tiles = ALL_TILES[:12]
results = []
start_time = time.time()
deadline = 6.0  # seconds

print('=' * 60)
print('SEQUENTIAL FLOOD PIPELINE')
print('=' * 60)
print(f'Processing {len(demo_tiles)} tiles. Deadline: {deadline}s')
print()

for i, tile in enumerate(demo_tiles):
    tile = download_tile(tile)
    result = generate_flood_mask(tile)
    results.append(result)
    save_to_postgis(result)

    elapsed = time.time() - start_time
    remaining = deadline - elapsed
    tiles_left = len(demo_tiles) - i - 1
    eta = tiles_left * 0.42

    fill = int((i + 1) / len(demo_tiles) * 28)
    bar = chr(9608) * fill + chr(9617) * (28 - fill)
    status = 'OK  ' if remaining > eta else 'LATE'
    print(f'[{bar}] {i+1:2d}/{len(demo_tiles)} | {elapsed:.1f}s | '
          f'ETA:{eta:.1f}s | left:{remaining:.1f}s | {status}')

    if remaining < 0 and i < len(demo_tiles) - 1:
        print()
        print('>>> FLOOD ARRIVED - SYSTEM NOT READY <<<')
        break

total = time.time() - start_time
print()
print(f'Total: {total:.2f}s | Processed: {len(results)}/{len(demo_tiles)}')

## Аналіз провалу

```
CPU Core Utilization:

  Core 1  [████████████████████████]  ~100% (виконує код)
  Core 2  [░░░░░░░░░░░░░░░░░░░░░░░░]    0% (простоює)
  Core 3  [░░░░░░░░░░░░░░░░░░░░░░░░]    0% (простоює)
  Core 4  [░░░░░░░░░░░░░░░░░░░░░░░░]    0% (простоює)

  Три чверті твого заліза - марнується.
```

### Де проблема?

**Single-core execution** - Python виконує інструкції по одній.  
Поки `generate_flood_mask(tile_1)` не завершиться - `tile_2` навіть не починається.

| Метрика | Значення |
|---------|----------|
| Задіяних cores | 1 з 4 |
| CPU efficiency | 25% |
| Bottleneck | Single-threaded CPython |

> **Думка розробника:** 'Треба додати threading! Потоки вирішать проблему!'

---

# ЕТАП 2 - Threading Illusion

```
┌──────────────────────────────────────────────────────────┐
│  JUNIOR DEV: 'Я додав ThreadPoolExecutor!'               │
│  SENIOR DEV: 'Яке завдання? I/O-bound чи CPU-bound?'    │
│  JUNIOR DEV: 'CPU-bound, SAR matrix operations...'       │
│  SENIOR DEV: (мовчить)                                      │
└──────────────────────────────────────────────────────────┘
```

## GIL: Чому threading не допомагає для CPU-bound

**Передбач результат ThreadPool з 4 workers для CPU-bound задачі:**

- А) 4x швидше (4 потоки = 4 cores)
- Б) Однаково (GIL блокує)
- В) Повільніше (context switching overhead)

In [ ]:
def process_tile_worker_func(tile_data):
    tile_id, region, raster_data = tile_data
    # CPU-bound: цей код буде боротись за GIL
    flood_px = 0

    for _ in range(40):
        flood_px += sum(
            1 for v in raster_data
            if v % 7 < 2
        )

    return {
        'tile_id': tile_id,
        'region': region,
        'flood_pixels': flood_px
    }
demo_data = [(t.tile_id, t.region, t.raster_data) for t in ALL_TILES[:8]]
n = len(demo_data)

# 1. Sequential baseline
start = time.time()
seq_r = [process_tile_worker_func(d) for d in demo_data]
seq_t = time.time() - start

# 2. ThreadPool (CPU-bound - expect GIL contention)
start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=CPU_CORES) as ex:
    thread_r = list(ex.map(process_tile_worker_func, demo_data))
thread_t = time.time() - start

print('GIL CONTENTION BENCHMARK')
print('=' * 50)
print(f'Sequential:   {seq_t:.2f}s  (1.00x baseline)')
ratio = thread_t / seq_t
verdict = 'SLOWER (GIL overhead!)' if ratio > 1.05 else ('same' if ratio > 0.95 else 'faster')
print(f'ThreadPool:   {thread_t:.2f}s  ({ratio:.2f}x) -> {verdict}')
print()
print('CPU Core Utilization (ThreadPool, CPU-bound):')
print(f'  Core 1  [{chr(9608)*20}{chr(9617)*4}]  ~100% (GIL holder)')
print(f'  Core 2  [{chr(9617)*5}{chr(9608)*3}{chr(9617)*16}]   ~15% (context switching)')
print(f'  Core 3  [{chr(9617)*7}{chr(9608)*2}{chr(9617)*15}]   ~10% (waiting for GIL)')
print(f'  Core 4  [{chr(9617)*9}{chr(9608)*2}{chr(9617)*13}]    ~8% (waiting for GIL)')

# GIL Contention Benchmark — Що тут відбувається

Цей benchmark показує:
# як ThreadPool поводиться на CPU-bound задачах.

---

# Головна ідея

Threading у Python:
- дуже хороший для I/O-bound задач
- але часто поганий для CPU-bound задач

через:
# GIL (Global Interpreter Lock)

---

# Worker function

```python
def process_tile_worker_func(tile_data):
````

Ця функція симулює:

* SAR analysis
* flood mask generation
* raster calculations

---

# Важливий рядок

```python
flood_px = sum(1 for v in raster_data if v % 7 < 2)
```

Тут Python:

* виконує цикл
* рахує значення
* постійно виконує Python bytecode

---

# Це CPU-bound workload

CPU:

* реально працює
* виконує обчислення
* НЕ чекає network
* НЕ чекає database

---

# Sequential baseline

```python
seq_r = [process_tile_worker_func(d) for d in demo_data]
```

Тут:

* один thread
* один interpreter
* один core

---

# Це baseline

```text id="jlwm7g"
Sequential = 1.00x
```

---

# ThreadPool version

```python
with concurrent.futures.ThreadPoolExecutor(max_workers=CPU_CORES)
```

Тут створюється:

* кілька threads
* всередині ОДНОГО Python process

---

# І тут головна проблема

## Усі threads ділять:

# один GIL

---
Він дозволяє:

* лише одному thread
* виконувати Python bytecode одночасно

---

# Тобто

Навіть якщо:

```text id="7jlwm1"
4 threads
```

Python НЕ дозволить:

```text id="9jlwm3"
4 simultaneous Python executions
```

---

# Що реально відбувається

```text id="2jlwm0"
Thread 1 -> GOT GIL
Thread 2 -> WAITING
Thread 3 -> WAITING
Thread 4 -> WAITING
```

---

# Через деякий час

```text id="6jlwm2"
Thread 1 -> RELEASE GIL
Thread 2 -> GOT GIL
```

---

# Threads починають

# боротися за interpreter lock

---

# CPU utilization diagram

```text
Core 1  ~100%
Core 2   ~15%
Core 3   ~10%
Core 4    ~8%
```

---

# Чому Core 1 майже повністю завантажений

Бо:

* один thread тримає GIL
* реально виконує Python code
* використовує CPU

---

# А інші cores?

Вони:

* mostly waiting
* scheduler activity
* context switching
* короткі bursts execution

---

# Тобто

Threads:

# НЕ працюють справді паралельно

---

# Що таке context switching

ОС:

* зупиняє один thread
* зберігає registers
* переключає execution
* активує інший thread

---

# Це overhead

CPU:

* витрачає час
* не на flood analysis
* а на management

---

# Чому ThreadPool може бути повільніший

Бо система отримує:

* GIL contention
* scheduler overhead
* context switching overhead

але:

* НЕ отримує true parallelism

---

# Головний insight

## Threading НЕ означає parallel execution

У CPython:

* CPU-bound threads
* часто виконуються майже послідовно

---

# Threading хороший для:

* HTTP requests
* database I/O
* file I/O
* network waiting

---

# Threading поганий для:

* raster math
* image processing
* ML training
* large Python loops
* heavy CPU calculations

---

# Для CPU-bound потрібен:

# ProcessPoolExecutor

Бо:

* кожен process має власний GIL
* ОС може розподілити processes по cores
* виникає справжній parallelism

---

# Головний висновок

Цей benchmark демонструє:

* що GIL обмежує CPU-bound threading
* чому ThreadPool ≠ automatic speedup
* чому workload type визначає architecture
* що CPU utilization важливіший за "кількість threads"

## Візуалізація роботи GIL

Ця симуляція показує, як Python threads "борються" за GIL (`Global Interpreter Lock`) під час CPU-bound задач.

У CPython:
- лише один thread може виконувати Python bytecode одночасно
- інші threads змушені чекати
- GIL передається між потоками по черзі

Тому навіть якщо створити:
```python
ThreadPoolExecutor(max_workers=4)
````

це НЕ означає:

```text id="jlwm5x"
4 simultaneous CPU computations
```

На CPU-bound workload threads:

* постійно чекають GIL
* створюють context switching overhead
* не дають справжнього multi-core parallelism

Ця timeline-візуалізація демонструє:

* який thread зараз тримає GIL
* які threads очікують
* як GIL переключається між потоками
* чому threading часто поганий для CPU-heavy задач



In [ ]:
# GIL Switching Timeline Visualization
print('GIL Timeline: 4 threads, CPU-bound task')
print()
print(f'Time   Thread-1         Thread-2         Thread-3         Thread-4')
print('-' * 75)

holder = 0
symbols = ['T1', 'T2', 'T3', 'T4']
for step in range(12):
    row = f't={step:<3}'
    for t in range(4):
        if t == holder:
            row += f'  [{chr(9608)*8} RUN  ]  '
        else:
            row += f'  [{chr(9617)*7} WAIT ]  '
    print(row)
    holder = (holder + 1) % 4

print()
print('Кожен момент часу: тiльки ОДИН потiк виконує Python code.')
print('GIL передається по черзi -> context switch -> overhead.')
print()
print('Коли GIL вiдпускається?')
print('  I/O-bound (socket.recv, file.read): ВIД ПУСКАЄТЬСЯ -> iншi потоки можуть працювати')
print('  CPU-bound (Python bytecode):        ТРИМАЄТЬСЯ   -> iнший потiк чекає')

## Чому GIL існує?

Python управляє пам'яттю через **reference counting**:

```python
my_list = [1, 2, 3]  # ref_count = 1
other = my_list      # ref_count = 2
del other            # ref_count = 1
del my_list          # ref_count = 0 -> garbage collected
```

Без GIL два потоки могли б **одночасно** декрементувати `ref_count`  
-> race condition -> use-after-free -> **segfault**.

### Правило вибору

| Тип задачi | GIL поведiнка | Iнструмент |
|------------|--------------|----------|
| I/O-bound (HTTP, файли, БД) | Вiдпускається пiд час wait | ThreadPoolExecutor |
| CPU-bound (математика, SAR) | Тримається весь час | ProcessPoolExecutor |

> **Висновок:** Для CPU-bound SAR processing нам потрiбен **справжнiй паралелiзм**.

---

# ЕТАП 3 - Real Parallelism

```
ІДЕЯ: Запустити кілька Python interpreters, кожен зі своїм GIL.

До fork():                  Після fork():

+──────────────+           +──────────────+    +──────────────+
│ Python Proc  │  fork()   │ Parent Proc  │    │  Child Proc  │
│ GIL          │ ────────> │ GIL (own)    │    │ GIL (copy!)  │
│ counter = 0  │           │ Core 1       │    │ Core 2       │
+──────────────+           +──────────────+    +──────────────+

Два GIL = два processes = два cores = СПРАВЖНІЙ паралелізм!
```

In [ ]:
import sys, os, pathlib, importlib.util

# Find the exact on-disk location of _flood_workers.py regardless of Jupyter CWD.
# Workers spawn a fresh interpreter — they inherit os.environ (OS-level) but
# sys.path propagation via multiprocessing is unreliable on Windows/Python 3.14.
# Setting PYTHONPATH guarantees workers find the module before any Python code runs.
_spec = importlib.util.find_spec('_flood_workers')
if _spec and _spec.origin:
    _lesson_dir = os.path.dirname(os.path.abspath(_spec.origin))
else:
    _lesson_dir = str(pathlib.Path('_flood_workers.py').resolve().parent)

if _lesson_dir not in sys.path:
    sys.path.insert(0, _lesson_dir)

_pp = os.environ.get('PYTHONPATH', '')
if _lesson_dir not in _pp:
    os.environ['PYTHONPATH'] = _lesson_dir + (os.pathsep + _pp if _pp else '')

from _flood_workers import (
    mp_process_tile,
    worker_heavy_ipc,
    worker_minimal_ipc,
    stage2_analyze,
    cpu_task,
    quick_task,
    sar_analyze,
)
print(f'_flood_workers imported OK')
print(f'Workers will find it at: {_lesson_dir}')
print(f'PYTHONPATH: {os.environ["PYTHONPATH"][:80]}')

In [ ]:
# Worker mp_process_tile вже iмпортовано з _flood_workers.py
# (ProcessPool може знайти його при spawn() на Windows)

demo_data = [(t.tile_id, t.region, t.raster_data) for t in ALL_TILES[:8]]

if __name__ == '__main__':
    print('MULTIPROCESSING FLOOD PIPELINE')
    print('=' * 55)
    print(f'ProcessPoolExecutor з {CPU_CORES} workers...')
    print()

    start = time.time()
    with concurrent.futures.ProcessPoolExecutor(max_workers=CPU_CORES) as ex:
        futures = {ex.submit(mp_process_tile, d): d[0] for d in demo_data}
        mp_results = []
        for fut in concurrent.futures.as_completed(futures):
            res = fut.result()
            mp_results.append(res)
            done = len(mp_results)
            total = len(demo_data)
            fill = int(done / total * 30)
            bar = chr(9608)*fill + chr(9617)*(30-fill)
            print(f'[{bar}] {done}/{total} PID={res["worker_pid"]} | '
                  f'{res["region"]}: {res["flood_pixels"]} flood px')

    mp_t = time.time() - start
    seq_estimate = len(demo_data) * TILE_PROCESS_TIME
    unique_pids = len(set(r['worker_pid'] for r in mp_results))

    print()
    print(f'Sequential estimate: {seq_estimate:.2f}s')
    print(f'ProcessPool actual:  {mp_t:.2f}s')
    print(f'Speedup: x{seq_estimate/mp_t:.2f}')
    print(f'Worker PIDs: {unique_pids} (кожен = окремий process)')

## Multiprocessing Flood Pipeline

Цей приклад демонструє:
# справжній multi-core parallelism у Python.

На відміну від `ThreadPoolExecutor`, де threads ділять один GIL,
`ProcessPoolExecutor` створює:
- окремі Python processes
- окремі interpreter-и
- окремий GIL для кожного process-а

Тому ОС може:
- розподілити workers по різних CPU cores
- виконувати flood analysis одночасно
- отримати реальний speedup для CPU-bound задач

---

# Що показує benchmark

```text id="jlwm3x"
Sequential estimate: 2.00s
ProcessPool actual: 0.40s
Speedup: x5.00
````

---

# Чому ProcessPool швидший

Кожен worker process:

* незалежний
* має власний interpreter
* виконується на окремому core

Тому:

* flood mask calculations
* raster analysis
* CPU-heavy loops

можуть працювати:

# паралельно

---

# Що показують PID

```text id="4jlwm5"
PID=44744
PID=46344
PID=45324
```

PID (`Process ID`) —
це унікальний ідентифікатор process-а в ОС.

Різні PID означають:

# різні processes

---

# Що тут важливо

На відміну від ThreadPool:

* workers НЕ борються за один GIL
* cores реально виконують compute одночасно
* CPU utilization стає значно ефективнішим

---

# Головний insight

Для:

* raster processing
* SAR analysis
* image processing
* ML workloads
* CPU-heavy computations

у Python зазвичай потрібен:

# ProcessPoolExecutor

а не ThreadPoolExecutor.

In [ ]:
# Workers iмпортованi з _flood_workers.py
# process_tile_worker_func = mp_process_tile (та сама логiка)

if __name__ == '__main__':
    n = 8
    data = [(t.tile_id, t.region, t.raster_data) for t in ALL_TILES[:n]]

    start = time.time()
    [mp_process_tile(d) for d in data]
    seq_t = time.time() - start

    start = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=CPU_CORES) as ex:
        list(ex.map(mp_process_tile, data))
    thr_t = time.time() - start

    start = time.time()
    with concurrent.futures.ProcessPoolExecutor(max_workers=CPU_CORES) as ex:
        list(ex.map(mp_process_tile, data))
    mp_t = time.time() - start

    print('BENCHMARK: CPU-bound SAR Processing')
    print('=' * 55)
    print(f'{"Method":<20} {"Time":>8} {"vs Seq":>10} {"CPU Cores":>12}')
    print('-' * 55)
    print(f'{"Sequential":<20} {seq_t:>7.2f}s {"1.00x":>10} {"1/"+str(CPU_CORES):>12}')
    t_ratio = thr_t / seq_t
    print(f'{"ThreadPool":<20} {thr_t:>7.2f}s {t_ratio:>9.2f}x {"1/"+str(CPU_CORES):>12}')
    mp_ratio = seq_t / mp_t
    print(f'{"ProcessPool":<20} {mp_t:>7.2f}s {mp_ratio:>9.2f}x {str(CPU_CORES)+"/"+str(CPU_CORES):>12}')
    print('=' * 55)

    print()
    print('CPU Core Utilization (ProcessPool):')
    for core in range(min(CPU_CORES, 4)):
        fill = 24 - core
        print(f'  Core {core+1}  [{chr(9608)*fill}{chr(9617)*(24-fill)}]  ~{100-core*3}%')

## Benchmark: CPU-bound SAR Processing

Цей benchmark порівнює:
- Sequential execution
- ThreadPoolExecutor
- ProcessPoolExecutor

для CPU-bound SAR processing задач.

---

# Sequential

```python
[mp_process_tile(d) for d in data]
```

Один process, один core, послідовне виконання.

---

# ThreadPool

```python
ThreadPoolExecutor(max_workers=CPU_CORES)
```

Threads ділять один GIL, тому:
- справжнього parallelism немає
- threads чекають GIL
- виникає overhead

Для CPU-bound задач ThreadPool часто:
- такий самий
- або повільніший за sequential

---

# ProcessPool

```python
ProcessPoolExecutor(max_workers=CPU_CORES)
```

Кожен worker:
- окремий process
- окремий interpreter
- окремий GIL

Тому ОС може розподілити tasks по cores і отримати:
# real multi-core parallelism

---

# Результат

```text id="0jlwm0"
Sequential:   2.00s
ProcessPool: 0.42s
Speedup: x4.81
```

---

# Головний висновок

Для:
- raster analysis
- image processing
- ML workloads
- CPU-heavy geospatial tasks

у Python зазвичай потрібен:
# ProcessPoolExecutor

а ThreadPool краще підходить для:
- network I/O
- database I/O
- file operations

---

# ЕТАП 4 - Pickle Disaster

```
┌──────────────────────────────────────────────────────────────┐
│  ТИЖДЕНЬ ПІЗНІШЕ. Prod deployment.                           │
│                                                              │
│  Система вже працює в production.                            │
│  Flood events обробляються в реальному часі.                 │
│                                                              │
│  Junior developer вирішує "оптимізувати" pipeline.           │
│                                                              │
│  DEV: 'Ми ж перейшли на multiprocessing.'                    │
│  DEV: 'Тепер все має літати.'                                │
│                                                              │
│  Він додає:                                                  │
│                                                              │
│  executor.map(process_tile, huge_numpy_arrays)               │
│                                                              │
│  де кожен tile = 120MB SAR raster                            │
│                                                              │
│  Через 10 хвилин...                                          │
│                                                              │
│  [CPU usage: 12%]                                            │
│  [RAM usage: 31 GB]                                          │
│  [System lag detected]                                       │
│  [Kernel swapping...]                                        │
│                                                              │
│  DEV: 'Чому multiprocessing ПОВІЛЬНІШИЙ?!'                   │
│                                                              │
│  [Sequential time:   8.2s]                                   │
│  [ProcessPool time: 47.3s]                                   │
│                                                              │
│  ARCHITECT мовчки відкриває htop.                            │
└──────────────────────────────────────────────────────────────┘
```


## Невидима стіна між процесами

На перший погляд здається, що `multiprocessing` просто запускає код на кількох ядрах CPU.
Але між процесами існує критично важлива особливість:

> Процеси НЕ мають спільної памʼяті.

Кожен process у Python:
- має власний address space
- власний heap
- власний Python interpreter
- власний GIL

## Parent process vs Child process

Коли Python запускає програму — ОС створює один process:

```text id="9jlwm5"
Parent Process
````

Коли використовується:

```python
ProcessPoolExecutor(max_workers=4)
```

Python створює нові processes:

```text id="3jlwm1"
Parent Process
├── Child Process 1
├── Child Process 2
└── Child Process 3
```

Кожен child process:

* має власну RAM
* власний Python interpreter
* власний GIL

Тому child process НЕ бачить Python-обʼєкти parent process-а напряму.

---

## Як передаються дані

```text id="7jlwm3"
Parent RAM
large_array
     │
     ▼
pickle.dumps()
     │
     ▼
OS Pipe / IPC
     │
     ▼
pickle.loads()
     │
     ▼
Child RAM
new large_array copy
```

Тобто multiprocessing передає НЕ pointer,
а КОПІЮ обʼєкта через serialization (`pickle`).



Тому child process не може просто "бачити" Python-обʼєкти батьківського процесу.

---

### Що очікує новачок

```python
executor.map(process_tile, huge_numpy_arrays)
````

Інтуїція підказує:

```text
Parent Process:
large_array ---> Child Process
```

Ніби Python просто передає pointer на memory.

Але так НЕ працює.

---

## Що реально відбувається

Передача обʼєкта між процесами складається з трьох дорогих етапів:

### 1. Serialization (`pickle.dumps`)

Python бере обʼєкт:

```python
large_array
```

і перетворює його у байтовий потік:

```python
pickle.dumps(large_array)
```

Для `numpy.ndarray` це означає:

* shape
* dtype
* metadata
* увесь memory buffer

копіюються у bytes.

---

### 2. IPC через OS Kernel

Отримані bytes проходять через:

* Pipe
* Socket
* Kernel buffer

Тобто дані фізично копіюються:

```text
User Space (Parent)
        ↓
Kernel Space
        ↓
User Space (Child)
```

Це НЕ pointer passing.

Це реальне копіювання памʼяті між address spaces.

---

### 3. Deserialization (`pickle.loads`)

Child process отримує bytes і виконує:

```python
pickle.loads(bytes)
```

Python:

* виділяє нову RAM
* створює новий object graph
* копіює bytes назад у memory

У результаті child process отримує ПОВНУ копію обʼєкта.

---

## Чому це стає катастрофою

Уявімо:

```text
1 tile = 120MB SAR raster
```

Якщо:

* 4 workers
* кілька tasks у черзі
* prefetching executor-а

то система може одночасно копіювати гігабайти даних через IPC.

---

## Що бачить ОС

Операційна система бачить не "flood analysis", а:

```text
ALLOCATE MEMORY
COPY MEMORY
COPY MEMORY
WAIT PIPE
ALLOCATE MEMORY
COPY MEMORY
```

---

## Чому CPU usage лише 12%

CPU не займається flood detection.

Він:

* серіалізує bytes
* копіює memory
* чекає IPC
* чекає RAM bandwidth
* працює з kernel buffers

Тому bottleneck стає:

| Було             | Стало        |
| ---------------- | ------------ |
| CPU computations | Memory + IPC |

---

## Чому Sequential виявився швидшим

Sequential pipeline:

* не копіює data між process-ами
* не використовує pickle
* не створює IPC overhead
* працює з локальною RAM

Тому:

```text
Sequential:
compute only

Multiprocessing:
serialize + copy + IPC + deserialize + compute
```

Іноді movement data дорожчий за самі обчислення.

---

## Production lesson

Для multiprocessing НЕ варто передавати:

* huge numpy arrays
* великі raster datasets
* великі object graphs

Краще передавати:

* filepath
* tile_id
* URL
* database key

Наприклад:

```python
executor.submit(process_tile, "/tiles/tile_001.tif")
```

Тоді child process:

* сам читає файл
* локально алокує RAM
* не створює IPC catastrophe

---

## Головний висновок

`multiprocessing` — це не "безкоштовний паралелізм".

Це компроміс між:

* parallel execution
* memory isolation
* IPC overhead
* serialization cost
* RAM bandwidth

У distributed systems дуже часто:

> передача даних коштує дорожче,
> ніж самі обчислення.


In [ ]:
import pickle

print('PICKLE OVERHEAD ANALYSIS')
print('=' * 55)

for size in [100, 1000, 10000, 50000]:
    data = list(range(size))
    raw_bytes = pickle.dumps(data)
    print(type(raw_bytes))
    size_kb = len(raw_bytes) / 1024

    start = time.time()
    for _ in range(5):
        b = pickle.dumps(data)
        _ = pickle.loads(b)
    pickle_ms = (time.time() - start) / 5 * 1000

    print(f'  {size:6d} elements ({size_kb:7.1f} KB) | '
          f'pickle+unpickle: {pickle_ms:.2f}ms')

print()
print('Для реального Sentinel-1 tile (300MB numpy array):')
print('  pickle time:     ~2-5 секунди')
print('  pipe transfer:   ~1-3 секунди')
print('  unpickle:        ~2-5 секунди')
print('  TOTAL overhead:  ~5-13 секунди на tile!')
print()
print('Якщо обробка = 0.3s -> overhead у 17-43x бiльший!')

## Що таке pickle у Python

`pickle` — це механізм серіалізації Python-обʼєктів.

Простими словами:

> `pickle` перетворює Python-обʼєкт у байтовий потік (`bytes`),
> щоб його можна було:
- зберегти у файл
- передати через мережу
- передати між процесами
- записати в pipe/socket

---

## Serialization → bytes

Наприклад:

```python
import pickle

data = [1, 2, 3]

raw = pickle.dumps(data)

print(type(raw))
````

Результат:

```python
<class 'bytes'>
```

Тобто Python список був перетворений у:

```text id="3lzhjlwm"
b'\x80\x04\x95...'
```

---

## Deserialization

Зворотний процес:

```python
original = pickle.loads(raw)
```

Python:

* читає bytes
* реконструює object graph
* створює нові Python objects у RAM

---

# Чому multiprocessing використовує pickle

Процеси ізольовані.

Child process НЕ бачить RAM батьківського процесу.

Тому:

```python
ProcessPoolExecutor()
```

не може просто передати:

* pointer
* reference
* memory address

між процесами.

---

## Тому Python робить:

```text id="4tjlwm"
Python object
    ↓
pickle.dumps()
    ↓
bytes
    ↓
OS Pipe
    ↓
pickle.loads()
    ↓
новий Python object
```

---

# Що показує цей benchmark

```python
for size in [100, 1000, 10000, 50000]:
```

Ти поступово збільшуєш:

* розмір object graph
* кількість елементів
* обсяг memory

---

## Для кожного size:

```python
data = list(range(size))
```

створюється список:

```python
[0, 1, 2, 3, ...]
```

---

## Потім:

```python
raw_bytes = pickle.dumps(data)
```

Python:

* обходить список
* читає кожен integer
* створює binary representation

---

## А:

```python
pickle.loads(b)
```

створює:

* новий список
* нові Python int objects
* нові pointers

---

# Чому це дорого

Serialization — це НЕ просто "зберегти файл".

Python повинен:

* обійти object graph
* перевірити типи
* серіалізувати metadata
* скопіювати memory
* створити bytes buffer

---

# А unpickle ще дорожчий

Бо Python:

* блокує RAM
* створює нові objects
* rebuild-ить структуру

---

# Важливий момент

## Python objects дуже важкі

Наприклад:

```python
int
```

у CPython — це НЕ просто число.

Це:

* object header
* refcount
* type pointer
* value

---

# Тому

```python
list(range(50000))
```

це:

* десятки тисяч Python objects
* pointers
* allocations

---

# Що показує benchmark

```text id="jlwm8f"
чим більший object
↓
тим більший pickle overhead
```

---

# Але тут ще маленькі дані

```python
50000 elements
```

це лише:

* сотні KB
* або кілька MB

---

# У реальному GeoAI pipeline

Sentinel-1 tile:

```text id="jlwm7f"
300MB numpy array
```

---

# І тоді починається production nightmare

Python повинен:

```text id="7jlwm2"
300MB RAM
↓
serialize
↓
copy
↓
kernel pipe
↓
copy
↓
deserialize
↓
allocate new RAM
```

---

# І тут головний insight

## Обчислення можуть бути дешевші за передачу даних

Наприклад:

| Operation          | Time |
| ------------------ | ---- |
| Flood mask compute | 0.3s |
| Pickle + IPC       | 8s   |

---

# Тобто

Система:

* не process-ить SAR
* а копіює memory

---

# Чому це важливо для architecture

Початківці думають:

```text id="5jlwm8"
більше processes = швидше
```

Але architect думає:

```text id="sjlwm3"
Скільки коштує movement data?
```

---

# Production pattern

НЕ:

```python
executor.submit(worker, huge_numpy_array)
```

---

# А:

```python
executor.submit(worker, tile_path)
```

---

# Тобто

Передавати:

* path
* ID
* reference

а НЕ:

* гігабайти RAM

---

# Це фундаментальний принцип distributed systems

> Move computation to data.
>
> NOT:
>
> Move data to computation.

---

# Головний висновок

`pickle` — це:

* serialization layer
* IPC mechanism foundation
* hidden multiprocessing cost

Іноді:

* serialization
* memory copies
* IPC overhead

коштують набагато дорожче,
ніж самі обчислення.


In [ ]:
# Workers worker_heavy_ipc, worker_minimal_ipc iмпортованi з _flood_workers.py

if __name__ == '__main__':
    n = 8
    heavy_args = [(f'tile_{i}', list(range(2000000))) for i in range(n)]
    minimal_ids = [f'tile_{i}' for i in range(n)]

    start = time.time()
    with concurrent.futures.ProcessPoolExecutor(max_workers=CPU_CORES) as ex:
        list(ex.map(worker_heavy_ipc, heavy_args))
    heavy_t = time.time() - start

    start = time.time()
    with concurrent.futures.ProcessPoolExecutor(max_workers=CPU_CORES) as ex:
        list(ex.map(worker_minimal_ipc, minimal_ids))
    minimal_t = time.time() - start

    print('IPC DATA TRANSFER IMPACT')
    print('=' * 45)
    print(f'Heavy IPC (20K elements):  {heavy_t:.3f}s')
    print(f'Minimal IPC (str only):    {minimal_t:.3f}s')
    if heavy_t > minimal_t:
        print(f'IPC overhead: x{heavy_t/minimal_t:.2f}')
    print()
    print('ПРАВИЛО: передавай мiнiмум через Process boundary.')
    print('Замiсть: executor.submit(process, huge_array)')
    print('Краще:   executor.submit(process_file, "/path/tile.tif")')

## Що показує цей код

Цей приклад демонструє:

> передавати великі дані між process-ами дорого.

---

# Код

```python
heavy_args = [(f'tile_{i}', list(range(2000000))) for i in range(n)]
minimal_ids = [f'tile_{i}' for i in range(n)]
````

Тут створюються ДВА типи задач.

---

# 1. Heavy IPC

```python
(f'tile_0', list(range(2000000)))
```

Тобто worker отримує:

* tile ID
* ВЕЛИКИЙ список чисел

Наприклад:

```python
('tile_0', [0,1,2,3,4,5,6,...])
```

---

# 2. Minimal IPC

```python
'tile_0'
```

Тут передається лише:

* маленький string

---

# Що таке IPC

IPC =

# Inter-Process Communication

тобто:

> передача даних між process-ами.

---

# Чому це важливо

Process-и ізольовані.

Вони НЕ мають спільної памʼяті.

---

# Тому Python змушений:

```text id="jlwm2y"
serialize object
↓
send bytes
↓
deserialize object
```

---

# Що робить цей рядок

```python
with concurrent.futures.ProcessPoolExecutor(max_workers=CPU_CORES) as ex:
```

Python створює:

* кілька process-ів
* окремі Python interpreter-и

Наприклад:

```text id="jlwm9d"
Main Process
├── Worker Process 1
├── Worker Process 2
├── Worker Process 3
└── Worker Process 4
```

---

# Далі

```python
list(ex.map(worker_heavy_ipc, heavy_args))
```

Python:

* бере `heavy_args`
* передає кожен argument у worker process

---

# І ось головний момент

```python
('tile_0', list(range(2000000)))
```

НЕ можна просто "показати" іншому process-у.

---

# Тому Python робить:

```text id="jlwm6v"
pickle.dumps(...)
```

тобто:

* перетворює object у bytes

---

# Потім

```text id="3jlwm5"
bytes
↓
OS pipe
↓
worker process
```

---

# А worker process робить

```text id="0jlwm1"
pickle.loads(...)
```

тобто:

* створює НОВИЙ object у своїй RAM

---

# Чому minimal IPC швидший

```python
'tile_0'
```

це дуже маленький object.

Передати його дешево.

---

# А

```python
list(range(2000000))
```

це:

* тисячі Python objects
* багато memory
* багато bytes

---

# Але чому overhead лише x3.0?

Бо:

```python
2000000 elements
```

це ще маленькі дані для сучасного компʼютера.

---

# У реальному GeoAI

Було б:

```python
300MB numpy array
```

---

# І тоді overhead був би величезний

Наприклад:

```text id="jlwm8z"
pickle: 3s
pipe transfer: 2s
unpickle: 3s
```

---

# Тобто

Process:

* ще НЕ почав flood analysis
* а вже витратив 8 секунд

---

# Найголовніша ідея

## multiprocessing хороший для:

* CPU-heavy задач
* коли дані маленькі
* коли computation дорогий

---

# multiprocessing поганий коли:

* data дуже великі
* треба багато IPC
* computation маленький

---

# Тому production pattern такий

## Погано

```python
executor.submit(worker, huge_numpy_array)
```

---

# Добре

```python
executor.submit(worker, "/tiles/tile_001.tif")
```

---

# Чому

Тоді process:

* сам читає файл
* без передачі гігабайтів через IPC

---

# Простий висновок

Цей код показує:

> передавати великі data між process-ами дорого.

Тому у multiprocessing:

* важливі не лише cores
* а ще:

  * RAM
  * serialization
  * IPC
  * movement of data

## Що не можна передати в Process?

Не всi Python обʼєкти можна serialize через pickle:

```python
import pickle

# Перевiрка:
try:
    pickle.dumps(my_object)
    print('OK: можна передати в Process')
except Exception as e:
    print(f'FAIL: {e}')
```

| Тип | Pickle | Чому |
|-----|--------|------|
| `int`, `str`, `list`, `dict` | OK | Серiалiзуються |
| `socket` | FAIL | OS-level handle |
| `lambda` | FAIL | Немає iменi для lookup |
| `generator` | FAIL | Lazy iterator без стану |
| `open file` | FAIL | OS file descriptor |
| `DB connection` | FAIL | TCP socket |

 **Аналогiя:**
> Так - Можна надiслати ключ вiд кiмнати другу в iншому мiстi?

> Нi - можна тiльки надiслати **план ключа**, щоб вiн зробив свiй.

# ЕТАП 5 — Architect Mode: Hybrid Pipeline

```

┌──────────────────────────────────────────────────────────────┐
│ HEAD ARCHITECT: 'Тепер ти розумiєш всi проблеми.'            │
│ 'Спроєктуй правильну систему.'                               │
│ Вимоги:                                                      │
│ - Download tiles паралельно (I/O-bound)                      │
│ - SAR analysis паралельно (CPU-bound)                        │
│ - Save results паралельно (I/O-bound)                        │
│ - Мiнiмальний IPC overhead                                   │
└──────────────────────────────────────────────────────────────┘
```




До цього моменту система вже пройшла через:
- sequential bottleneck
- GIL problem
- multiprocessing overhead
- pickle disaster
- IPC bottlenecks

Тепер задача:
# побудувати нормальну production architecture.

---

# Головна проблема

У flood pipeline різні етапи мають різний тип навантаження.

| Stage | Тип workload |
|---|---|
| Download tiles | I/O-bound |
| SAR analysis | CPU-bound |
| Save to PostGIS | I/O-bound |

І саме тому:
# не можна використовувати один executor для всього.

---

# STAGE 1 — Download

```text id="jlwm1p"
DOWNLOAD
(I/O-bound)
````

Тут:

* HTTP requests
* очікування Copernicus Hub
* network latency

CPU майже нічого не рахує.

Під час:

```python
requests.get(...)
```

thread просто чекає відповідь від сервера.

---

# Тому тут ThreadPool

```text id="9jlwm2"
ThreadPoolExecutor(10 workers)
```

ідеальний.

Threads можуть:

* одночасно чекати network
* не блокувати CPU
* ефективно використовувати I/O wait time

---

# STAGE 2 — Processing

```text id="jlwm4r"
PROCESS
(CPU-bound)
```

Тут:

* raster calculations
* thresholding
* flood mask generation
* pixel operations

CPU реально виконує обчислення.

---

# Тут threading вже поганий

Через:

# GIL

Threads почнуть:

* боротися за interpreter lock
* створювати context switching overhead
* працювати майже послідовно

---

# Тому тут ProcessPool

```text id="5jlwm3"
ProcessPoolExecutor(cpu_count())
```

Кожен process:

* має власний interpreter
* має власний GIL
* працює на окремому core

---

# STAGE 3 — Save

```text id="1jlwm9"
SAVE
(I/O-bound)
```

Тут:

* INSERT into PostGIS
* disk I/O
* DB latency
* network wait

CPU знову майже нічого не рахує.

---

# Тому Save → ThreadPool

Threading добре працює для:

* database I/O
* file I/O
* network operations

---

# Queue coordination

```text id="8jlwm7"
DOWNLOAD → QUEUE → PROCESS → QUEUE → SAVE
```

Queue:

* зберігає tasks між stages
* дозволяє stages працювати незалежно
* buffer-изує різницю у швидкості

---

# Наприклад

Download:

```text id="4jlwm4"
20 tiles/sec
```

Processing:

```text id="2jlwm1"
4 tiles/sec
```

---

# Що станеться

Download stage:

* генерує tiles швидше
* queue росте
* processing workers поступово забирають tasks

---

# Це producer-consumer architecture

## Download stage

```text id="6jlwm3"
Producer
```

створює tasks.

---

# Process stage

```text id="7jlwm5"
Consumer
```

споживає tasks.

---

# Найважливіша оптимізація

```text id="0jlwm8"
Stage 2 отримує тільки tile_id
```

---

# Чому це критично

Поганий варіант:

```python
executor.submit(process_tile, huge_numpy_array)
```

---

# Що буде

Python:

* pickle.dumps()
* pipe transfer
* pickle.loads()

---

# Для великих raster

це:

* секунди overhead
* RAM explosion
* IPC bottleneck

---

# Хороший варіант

```python
executor.submit(process_tile, tile_path)
```

---

# Тоді process

сам:

* відкриває GeoTIFF
* читає локальні дані
* process-ить без IPC catastrophe

---

# Головний принцип

## Move computation to data

НЕ:

## Move data to computation

---

# Що робить ця architecture

```text id="9jlwm6"
I/O → Threads
CPU → Processes
I/O → Threads
```

---

# Результат

Система:

* використовує всі cores
* не блокується на GIL
* не перевантажує IPC
* не копіює гігабайти RAM
* ефективно перекриває latency

---

In [ ]:
# stage2_analyze iмпортовано з _flood_workers.py
# stage1_download i stage3_save - ThreadPool (не потребують spawn fix)

def stage1_download(tile_id):
    # I/O-bound: ThreadPool - нема потреби у _flood_workers.py
    time.sleep(TILE_DOWNLOAD_TIME)
    return {'tile_id': tile_id, 'local_path': f'/cache/{tile_id}.tif'}

def stage3_save(result):
    # I/O-bound: ThreadPool - нема потреби у _flood_workers.py
    time.sleep(0.05)
    return result['tile_id']

if __name__ == '__main__':
    n = 16
    tile_ids = [f'S1A_IW_{i:04d}' for i in range(n)]

    print('HYBRID FLOOD PIPELINE')
    print('=' * 55)
    start_total = time.time()

    s1 = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as ex:
        downloaded = list(ex.map(stage1_download, tile_ids))
    s1_t = time.time() - s1
    print(f'Stage 1 Download  (ThreadPool): {s1_t:.2f}s | {n} tiles')

    s2 = time.time()
    with concurrent.futures.ProcessPoolExecutor(max_workers=CPU_CORES) as ex:
        processed = list(ex.map(stage2_analyze, downloaded))
    s2_t = time.time() - s2
    unique_pids = len(set(r['worker_pid'] for r in processed))
    print(f'Stage 2 Analyze   (ProcessPool): {s2_t:.2f}s | {unique_pids} workers')

    s3 = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as ex:
        saved = list(ex.map(stage3_save, processed))
    s3_t = time.time() - s3
    print(f'Stage 3 Save      (ThreadPool): {s3_t:.2f}s | {len(saved)} saved')

    total_t = time.time() - start_total
    seq_est = n * (TILE_DOWNLOAD_TIME + TILE_PROCESS_TIME + 0.05)
    print()
    print(f'Total: {total_t:.2f}s | Sequential est: {seq_est:.2f}s | Speedup: x{seq_est/total_t:.2f}')
    alerts = [r for r in processed if r['flood_pct'] > 15]
    print(f'FLOOD ALERTS: {len(alerts)}/{n} regions critical')
    print('FLOOD MAP GENERATED. ДСНС отримала данi вчасно.')

## Live Pipeline Telemetry
Що тут реально відбувається

Це вже не просто multiprocessing demo.

Це:
# production-style concurrent pipeline

де різні stages:
- працюють паралельно
- мають різний workload
- використовують різні executor-и
- взаємодіють через futures

---

# Головна architecture

```text id="3jlwm4"
DOWNLOAD → PROCESS → SAVE

In [ ]:
# stage2_analyze iмпортовано з _flood_workers.py
# stage1_download, stage3_save визначенi у попереднiй клiтинцi

if __name__ == '__main__':
    n = 20
    tile_ids = [f'S1A_IW_{i:04d}' for i in range(n)]

    print('LIVE FLOOD RESPONSE TELEMETRY')
    print('=' * 60)

    stats = {'dl': 0, 'proc': 0, 'saved': 0, 'alerts': 0}
    start = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as dl_ex, \
         concurrent.futures.ProcessPoolExecutor(max_workers=CPU_CORES) as proc_ex, \
         concurrent.futures.ThreadPoolExecutor(max_workers=4) as save_ex:

        dl_futures = {dl_ex.submit(stage1_download, tid): tid for tid in tile_ids}
        proc_futures = {}
        save_futures = []

        for dl_f in concurrent.futures.as_completed(dl_futures):
            info = dl_f.result()
            stats['dl'] += 1
            pf = proc_ex.submit(stage2_analyze, info)
            proc_futures[pf] = info['tile_id']

        for proc_f in concurrent.futures.as_completed(proc_futures):
            res = proc_f.result()
            stats['proc'] += 1
            if res['flood_pct'] > 15:
                stats['alerts'] += 1
            sf = save_ex.submit(stage3_save, res)
            save_futures.append(sf)

        for sf in concurrent.futures.as_completed(save_futures):
            sf.result()
            stats['saved'] += 1

    elapsed = time.time() - start

    print(f'Tiles downloaded:  {stats["dl"]}')
    print(f'Tiles processed:   {stats["proc"]}')
    print(f'Tiles saved:       {stats["saved"]}')
    print(f'Flood alerts:      {stats["alerts"]}')
    print(f'Total time:        {elapsed:.2f}s')
    print(f'Throughput:        {stats["proc"]/elapsed:.1f} tiles/sec')
    print()
    print('STATUS: FLOOD MAP DELIVERED TO ДСНС')

---

# CHAOS ENGINEERING: Production Failures

```
'В продакшнi зламається все, що може зламатись.' - Murphy
```

## Failure 1 - Windows Spawn Bomb

**Симптоми:** macOS/Linux - OK. Windows - нескiнченнi вiкна або `RuntimeError`.

```python
# ЗЛАМАНО (Windows spawn bomb):
from multiprocessing import Process

def worker(): print('Worker!')

p = Process(target=worker)
p.start()  # Windows: spawn -> import script -> Process().start() -> ...
p.join()

# ВИПРАВЛЕНО:
if __name__ == '__main__':
    p = Process(target=worker)
    p.start()
    p.join()
```

**Чому?** Windows не має `fork()`. Використовує `spawn()`:  
1. Запускає новий interpreter  
2. **Iмпортує** скрипт (виконує module-level код)  
3. Якщо module-level мiстить `Process().start()` без guard - рекурсiя

In [ ]:
# quick_task iмпортовано з _flood_workers.py
import  multiprocessing as mp
if __name__ == '__main__':
    print('FAILURE 2: Zombie Processes')
    print('-' * 40)

    # ЗЛАМАНО: без join()
    procs = []
    for i in range(5):
        p = mp.Process(target=quick_task, args=(f'tile_{i}',))
        p.start()
        procs.append(p)

    time.sleep(0.2)
    not_joined = sum(1 for p in procs if p.exitcode is not None)
    print(f'{not_joined} процесiв завершились але p.join() не викликано')
    print('-> ОС зберiгає tombstone запис у таблицi процесiв')

    print()
    print('ВИПРАВЛЕННЯ:')
    for p in procs:
        p.join()
    print('join() виконано -> zombies прибранi -> OK')


In [ ]:
# cpu_task iмпортовано з _flood_workers.py

if __name__ == '__main__':
    print('FAILURE 3: CPU Oversubscription')
    print('-' * 40)
    print(f'CPU cores: {CPU_CORES}')
    print()

    WIN_MAX = 61  # Windows: ProcessPoolExecutor hard limit
    test_data = list(range(CPU_CORES * 2))
    bench = {}

    n_optimal = CPU_CORES
    n_over = min(CPU_CORES * 4, WIN_MAX)

    for n_w, label in [(n_optimal, 'optimal'), (n_over, 'oversubscribed')]:
        start = time.time()
        with concurrent.futures.ProcessPoolExecutor(max_workers=n_w) as ex:
            list(ex.map(cpu_task, test_data))
        elapsed = time.time() - start
        bench[n_w] = elapsed
        bar = chr(9608) * int(elapsed * 15)
        print(f'  {n_w:3d} workers ({label:15s}): {elapsed:.3f}s [{bar}]')

    print()
    if bench[n_over] > bench[n_optimal]:
        pct = round((bench[n_over] / bench[n_optimal] - 1) * 100, 0)
        print(f'Oversubscription ({n_over} workers) повiльнiший на {pct:.0f}%')
    print(f'ПРАВИЛО: max_workers = os.cpu_count() = {CPU_CORES} для CPU-bound')

---

# Практичнi Вправи

## Вправа 1: Debug the Flood Pipeline (Рiвень: Basic)

Знайди 3 баги в кодi нижче.

In [ ]:
# FLOOD PIPELINE З БАГАМИ
import multiprocessing as mp
import concurrent.futures

def sar_analyze(tile_id):
    time.sleep(0.1)
    return {'tile': tile_id, 'flood': abs(hash(tile_id)) % 30}

# Bug 1: Missing __name__ guard (RuntimeError на Windows)
# executor = concurrent.futures.ProcessPoolExecutor(max_workers=4)
# results = list(executor.map(sar_analyze, ['t1','t2','t3']))

# Bug 2: Передача великих даних через IPC (ПОВIЛЬНО)
# def bad_worker(args):
#     tile_id, huge_list = args
#     return sum(huge_list)
# bad_args = [(f't{i}', list(range(100_000))) for i in range(4)]
# with concurrent.futures.ProcessPoolExecutor() as ex:
#     results = list(ex.map(bad_worker, bad_args))

# Bug 3: Zombie processes
# procs = []
# for i in range(5):
#     p = mp.Process(target=sar_analyze, args=(f't{i}',))
#     p.start()
#     procs.append(p)
# # ЗАБУЛИ: for p in procs: p.join()

# TODO: Виправ кожен bug та поясни:
# Bug 1: додай if __name__ == '__main__':
# Bug 2: передавай тiльки tile_id, не самi данi
# Bug 3: додай for p in procs: p.join()

if __name__ == '__main__':
    print('Розкоментуй баги по одному, виправ i запусти.')

## Вправа 2: Architect Challenge (Рiвень: Advanced)

Спроєктуй pipeline для:

- **Download:** 50 tiles x 0.3s (I/O-bound)
- **SAR Analysis:** 50 tiles x 0.5s (CPU-bound)
- **Save:** 50 tiles x 0.1s (I/O-bound)

**Обмеження:** 4 CPU cores

**Питання:**
1. Який Executor для кожної стадiї?
2. Скiльки workers?
3. Теоретичний мiнiмальний час?

Реалiзуй та заміряй реальний vs теоретичний час.

In [ ]:
if __name__ == '__main__':
    N = 20  # скорочено для демо

    def dl(tid): time.sleep(0.15); return {'tile_id': tid}
    def analyze(info): time.sleep(0.2); return {'tile_id': info['tile_id'], 'flood': 5}
    def save_r(res): time.sleep(0.05); return res['tile_id']

    tiles = [f'tile_{i}' for i in range(N)]

    # Sequential baseline
    start = time.time()
    for tid in tiles:
        r1 = dl(tid); r2 = analyze(r1); save_r(r2)
    seq = time.time() - start
    print(f'Sequential: {seq:.2f}s')

    theoretical_min = max(
        N * 0.15 / 10,   # download з 10 thread workers
        N * 0.2 / CPU_CORES,  # analyze з cpu_count processes
        N * 0.05 / 5     # save з 5 thread workers
    )
    print(f'Theoretical min (staged): {theoretical_min:.2f}s')
    print()

    # TODO: реалiзуй hybrid pipeline тут
    # Пiдказка: використовуй 3 окремi executor блоки
    # або pipeline через as_completed
    print('Реалiзуй hybrid pipeline i порiвняй з sequential!')

---

# Summary: Flood Response Architecture

## Що ми вивчили через катастрофу?

| Етап | Час (16 tiles) | Урок |
|------|----------------|------|
| Sequential | ~6.7s | 1 core з 4 - марнування |
| Threading CPU-bound | ~7.0s | GIL блокує - overhead |
| ProcessPool | ~1.9s | Справжнiй паралелiзм |
| Naive ProcessPool (великi данi) | ~8.0s | Pickle вбиває |
| Hybrid Pipeline | ~2.1s | Правильна архiтектура |

## 5 Золотих Правил

```
1. CPU-bound задача?
      -> ProcessPoolExecutor(max_workers=os.cpu_count())

2. I/O-bound задача?
      -> ThreadPoolExecutor(max_workers=10-50)

3. Передача даних мiж процесами?
      -> Тiльки str, int, tuple
      -> Перевiрка: pickle.dumps(obj) перед використанням
      -> Нiколи: socket, lambda, generator, DB connection

4. Запуск на Windows?
      -> if __name__ == '__main__': ОБОВ'ЯЗКОВО

5. Запускаєш процеси?
      -> Завжди p.join() або 'with' context manager
      -> Zombie processes = витiк OS resources
```

## Дерево рiшень

```
Задача:
  +-- I/O-bound (мережа, файли, БД)
  |      +-> ThreadPoolExecutor(max_workers=10-50)
  |
  +-- CPU-bound (математика, SAR analysis, ML)
         +-> ProcessPoolExecutor(max_workers=os.cpu_count())
             +-- Данi прості (str, int)?   -> OK
             +-- Данi складнi (object)?    -> pickle.dumps check
             +-- Windows?                  -> __name__ guard
```

## Що далi - Урок 34: asyncio

Третiй пiдхiд до конкурентностi:
- `threading`: OS-level context switching
- `multiprocessing`: OS-level process forking  
- `asyncio`: кооперативнi coroutines, event loop

Iдеальний для: 10,000+ одночасних HTTP з'єднань без жодного потоку.